In [7]:
# Run this in Google Colab to prepare all data files
import pandas as pd
import numpy as np
import os

print("="*60)
print("📊 PREPARING EXECUTIVE FINANCIAL DASHBOARD DATA")
print("="*60)

# Set up paths
data_path = '/content/drive/MyDrive/data-analytics-projects/executive-financial-dashboard/data'
os.makedirs(data_path, exist_ok=True)

# Generate comprehensive financial data
np.random.seed(42)

# Create 3 years of monthly data (2023-2025)
dates = pd.date_range(start='2023-01-01', end='2025-12-31', freq='MS')
rows_per_month = 20
total_rows = len(dates) * rows_per_month

print(f"\n📈 Generating {total_rows:,} rows of financial data...")

departments = ['Engineering', 'Sales', 'Marketing', 'Operations', 'Finance']
regions = ['North America', 'Europe', 'Asia', 'South America', 'APAC']

# Main financial data
df = pd.DataFrame({
    'Date': np.repeat(dates, rows_per_month),
    'Year': np.repeat(dates.year, rows_per_month),
    'Month_Num': np.repeat(dates.month, rows_per_month),
    'Month': np.repeat(dates.strftime('%B'), rows_per_month),
    'Quarter': np.repeat([f'Q{q}' for q in dates.quarter], rows_per_month),
    'Department': np.random.choice(departments, total_rows, p=[0.25, 0.35, 0.20, 0.12, 0.08]),
    'Region': np.random.choice(regions, total_rows, p=[0.40, 0.25, 0.20, 0.10, 0.05]),
    'Revenue': np.random.normal(150000, 40000, total_rows).astype(int).clip(80000, 280000),
    'COGS': np.random.normal(45000, 15000, total_rows).astype(int).clip(20000, 90000),
    'Operating_Expenses': np.random.normal(35000, 12000, total_rows).astype(int).clip(15000, 70000),
    'Budget': np.random.normal(170000, 35000, total_rows).astype(int).clip(90000, 300000),
    'Marketing_Spend': np.random.normal(25000, 8000, total_rows).astype(int).clip(10000, 50000),
    'Headcount': np.random.randint(50, 500, total_rows),
})

# Add calculated fields
df['Gross_Profit'] = df['Revenue'] - df['COGS']
df['Net_Profit'] = df['Gross_Profit'] - df['Operating_Expenses']
df['Profit_Margin'] = (df['Net_Profit'] / df['Revenue'] * 100).round(2)
df['Variance'] = df['Revenue'] - df['Budget']
df['Variance_Pct'] = (df['Variance'] / df['Budget'] * 100).round(2)

# Add YoY calculations
df = df.sort_values(['Department', 'Region', 'Date'])
df['Prior_Year_Revenue'] = df.groupby(['Department', 'Region'])['Revenue'].shift(12)
df['Revenue_YoY_Growth'] = ((df['Revenue'] - df['Prior_Year_Revenue']) / df['Prior_Year_Revenue'] * 100).round(2)
df = df.fillna(0)

# Save main data
main_file = os.path.join(data_path, 'financial_data.csv')
df.to_csv(main_file, index=False)
print(f"✅ Saved: financial_data.csv ({len(df):,} rows)")

# Create monthly summary
monthly = df.groupby(['Year', 'Month', 'Month_Num', 'Quarter']).agg({
    'Revenue': 'sum',
    'Net_Profit': 'sum',
    'Profit_Margin': 'mean',
    'Budget': 'sum',
    'Variance': 'sum',
    'Marketing_Spend': 'sum',
    'Headcount': 'mean'
}).reset_index().sort_values(['Year', 'Month_Num'])

monthly_file = os.path.join(data_path, 'monthly_summary.csv')
monthly.to_csv(monthly_file, index=False)
print(f"✅ Saved: monthly_summary.csv ({len(monthly)} rows)")

# Create department summary
dept = df.groupby(['Year', 'Department']).agg({
    'Revenue': 'sum',
    'Net_Profit': 'sum',
    'Profit_Margin': 'mean',
    'Headcount': 'mean',
    'Revenue_YoY_Growth': 'mean'
}).reset_index()

dept_file = os.path.join(data_path, 'department_summary.csv')
dept.to_csv(dept_file, index=False)
print(f"✅ Saved: department_summary.csv")

# Create region summary
region = df.groupby(['Year', 'Region']).agg({
    'Revenue': 'sum',
    'Net_Profit': 'sum',
    'Profit_Margin': 'mean'
}).reset_index()

region_file = os.path.join(data_path, 'region_summary.csv')
region.to_csv(region_file, index=False)
print(f"✅ Saved: region_summary.csv")

# Create budget variance summary
budget_variance = df.groupby(['Year', 'Department']).agg({
    'Budget': 'sum',
    'Revenue': 'sum',
    'Variance': 'sum',
    'Variance_Pct': 'mean'
}).reset_index()

budget_file = os.path.join(data_path, 'budget_variance.csv')
budget_variance.to_csv(budget_file, index=False)
print(f"✅ Saved: budget_variance.csv")

print("\n" + "="*60)
print("✅ ALL DATA FILES CREATED!")
print("="*60)
print(f"\n📁 Location: {data_path}")
print("\n📄 Files created:")
print("   • financial_data.csv - Main transactions (for detailed analysis)")
print("   • monthly_summary.csv - Monthly aggregates (for trends)")
print("   • department_summary.csv - Department performance")
print("   • region_summary.csv - Regional analysis")
print("   • budget_variance.csv - Budget tracking")

📊 PREPARING EXECUTIVE FINANCIAL DASHBOARD DATA

📈 Generating 720 rows of financial data...
✅ Saved: financial_data.csv (720 rows)
✅ Saved: monthly_summary.csv (36 rows)
✅ Saved: department_summary.csv
✅ Saved: region_summary.csv
✅ Saved: budget_variance.csv

✅ ALL DATA FILES CREATED!

📁 Location: /content/drive/MyDrive/data-analytics-projects/executive-financial-dashboard/data

📄 Files created:
   • financial_data.csv - Main transactions (for detailed analysis)
   • monthly_summary.csv - Monthly aggregates (for trends)
   • department_summary.csv - Department performance
   • region_summary.csv - Regional analysis
   • budget_variance.csv - Budget tracking


In [8]:
# Step 4: Verify your data
import pandas as pd
import os

data_path = '/content/drive/MyDrive/data-analytics-projects/executive-financial-dashboard/data'

print("="*60)
print("🔍 VERIFYING YOUR DATA")
print("="*60)

# List all files
files = os.listdir(data_path)
csv_files = [f for f in files if f.endswith('.csv')]

print(f"\n📁 Files in your data folder:")
for file in csv_files:
    file_path = os.path.join(data_path, file)
    df = pd.read_csv(file_path)
    print(f"\n📊 {file}")
    print(f"   Rows: {len(df):,}")
    print(f"   Columns: {list(df.columns)[:5]}...")
    print(f"   Preview:")
    print(df.head(2))
    print("-" * 40)

print("\n✅ All data is ready!")

🔍 VERIFYING YOUR DATA

📁 Files in your data folder:

📊 financial_data.csv
   Rows: 720
   Columns: ['Date', 'Year', 'Month_Num', 'Month', 'Quarter']...
   Preview:
         Date  Year  Month_Num Month Quarter   Department Region  Revenue  \
0  2023-05-01  2023          5   May      Q2  Engineering   APAC   158956   
1  2023-07-01  2023          7  July      Q3  Engineering   APAC    95207   

    COGS  Operating_Expenses  Budget  Marketing_Spend  Headcount  \
0  42477               22609  186035            21893        464   
1  63928               41984  141901            18781        257   

   Gross_Profit  Net_Profit  Profit_Margin  Variance  Variance_Pct  \
0        116479       93870          59.05    -27079        -14.56   
1         31279      -10705         -11.24    -46694        -32.91   

   Prior_Year_Revenue  Revenue_YoY_Growth  
0                 0.0                 0.0  
1                 0.0                 0.0  
----------------------------------------

📊 monthly_summ

In [9]:
# Step 5: Open Google Drive folder
from google.colab import output
output.serve_kernel_port_as_iframe(8080)

# Show the path
print("📁 Your data is in Google Drive at:")
print("/content/drive/MyDrive/data-analytics-projects/executive-financial-dashboard/data")
print("\n💡 To view files:")
print("1. Click on the 'Files' folder icon on the left sidebar")
print("2. Navigate to: drive/MyDrive/data-analytics-projects/executive-financial-dashboard/data")
print("3. You'll see all your CSV files!")

<IPython.core.display.Javascript object>

📁 Your data is in Google Drive at:
/content/drive/MyDrive/data-analytics-projects/executive-financial-dashboard/data

💡 To view files:
1. Click on the 'Files' folder icon on the left sidebar
2. Navigate to: drive/MyDrive/data-analytics-projects/executive-financial-dashboard/data
3. You'll see all your CSV files!
